# Minería de Datos sobre Capa Oro (Sistemas Solares)

Este notebook desarrolla un análisis completo de **minería de datos** sobre la información de telemetría de los controladores de carga Epever.

### Objetivos
1. **Generar y explorar variables objetivo** (ej. Alertas de batería baja).
2. **Analizar combinaciones de condiciones operativas**.
3. **Agrupar perfiles de días (Clustering)** usando K-Means.
4. **Visualizar patrones** usando reducción de dimensionalidad (PCA y t-SNE).

## Importar las librerías necesarias

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import warnings
warnings.filterwarnings('ignore')

MINERIA_DIR = Path('../data/mineria')
MINERIA_DIR.mkdir(parents=True, exist_ok=True)

print("Librerías importadas.")

## Carga de datos

Cargamos los datasets generados en la **Capa Oro**.

In [ ]:
file_diario = '../data/oro/dataset_oro_diario_solar.csv'
file_horario = '../data/oro/dataset_oro_horario_solar.csv'

try:
    df_diario = pd.read_csv(file_diario)
    df_base = pd.read_csv(file_horario)
    print(f"Datos cargados. Diario: {df_diario.shape}, Horario: {df_base.shape}")
except FileNotFoundError:
    print("No se encontraron los archivos. Asegúrate de ejecutar el notebook de Capa Oro antes.")
    df_diario = pd.DataFrame()


## Generación de la variable objetivo: `ALERTA_BATERIA`

En sistemas aislados (off-grid), una descarga profunda acorta la vida útil de las baterías. Definiremos una alerta heurística:

- **1 (Alerta Crítica)** si el SOC promedio (`battery_soc_%_MEAN`) < 50% **O** el voltaje mínimo del día (`battery_voltage_V_MIN`) < 12.0V.
- **0 (Operación Normal)** en caso contrario.

In [ ]:
if not df_diario.empty:
    # Definir umbrales
    soc_minimo = 50.0
    voltaje_critico = 12.0
    
    # Crear variable binaria
    df_diario['ALERTA_BATERIA'] = ((df_diario['battery_soc_%_MEAN'] < soc_minimo) | 
                                   (df_diario['battery_voltage_V_MIN'] < voltaje_critico)).astype(int)
    
    # Guardar en la carpeta de minería
    df_diario.to_csv(MINERIA_DIR / 'dataset_mineria_diario.csv', index=False)
    print("Variable ALERTA_BATERIA generada y dataset guardado.")


## Exploración de la variable `ALERTA_BATERIA`

In [ ]:
if not df_diario.empty:
    plt.figure(figsize=(6,4))
    df_diario['ALERTA_BATERIA'].value_counts().plot(kind='bar', color=['#2ca02c', '#d62728'])
    plt.title("Distribución de días con Alerta de Batería")
    plt.xlabel("0: Normal | 1: Alerta")
    plt.ylabel("Cantidad de Días")
    plt.xticks(rotation=0)
    plt.show()


### Heatmap de Correlación General
Buscamos entender qué factores operacionales correlacionan más con nuestra variable de alerta.

In [ ]:
if not df_diario.empty:
    # Variables relevantes
    vars_corr = [
        'pv_power_W_MEAN', 'controller_temp_C_MEAN', 'energy_generated_today_kWh_MAX',
        'battery_voltage_RANGE', 'ALERTA_BATERIA'
    ]
    
    # Filtrar columnas que sí existan en el df
    vars_corr = [v for v in vars_corr if v in df_diario.columns]
    
    plt.figure(figsize=(8,6))
    sns.heatmap(df_diario[vars_corr].corr(), annot=True, cmap='RdBu_r', fmt=".2f", vmin=-1, vmax=1)
    plt.title("Correlación de Variables Operativas y Alertas")
    plt.tight_layout()
    plt.show()


## Combinaciones Discretas de Operación
Dado que los algoritmos como Apriori sufren con datos continuos temporales, discretizamos las variables de Potencia PV y Temperatura para ver qué combinaciones generan más alertas.

In [ ]:
if not df_diario.empty:
    df_assoc = df_diario.copy()
    
    # Discretización en categorías
    df_assoc['PV_CAT'] = pd.qcut(df_assoc['pv_power_W_MEAN'], q=2, labels=['Baja_Gen', 'Alta_Gen'])
    df_assoc['TEMP_CAT'] = pd.qcut(df_assoc['controller_temp_C_MEAN'], q=2, labels=['Frio', 'Calor'])
    
    resumen = df_assoc.groupby(['PV_CAT', 'TEMP_CAT'])['ALERTA_BATERIA'].agg(['count', 'mean']).reset_index()
    resumen.rename(columns={'count': 'Total_Dias', 'mean': 'Prob_Alerta'}, inplace=True)
    resumen['Prob_Alerta'] = (resumen['Prob_Alerta'] * 100).round(1).astype(str) + '%'
    
    display(resumen)


## Clustering de Días Operativos (K-Means)

Se aplica **K-Means** para agrupar días según su comportamiento. Esto nos permite identificar perfiles como "Día de alta carga y mucho sol" vs "Día nublado y batería baja".

In [ ]:
if not df_diario.empty:
    # Seleccionamos variables de trabajo
    cols_clust = ['pv_power_W_MEAN', 'battery_soc_%_MEAN', 'battery_voltage_RANGE', 'controller_temp_C_MEAN']
    cols_clust = [c for c in cols_clust if c in df_diario.columns]
    
    X_clust = df_diario[cols_clust].dropna()
    
    # Escalado
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_clust)
    
    # Aplicar K-Means (elegimos 3 clusters como ejemplo de perfiles diarios)
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    df_diario.loc[X_clust.index, 'CLUSTER'] = kmeans.fit_predict(X_scaled)
    
    print("Distribución de días por Cluster:")
    print(df_diario['CLUSTER'].value_counts())


### Visualización con PCA
Para entender los clústeres, reducimos la dimensionalidad a 2 dimensiones (Componentes Principales).

In [ ]:
if not df_diario.empty:
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
    df_pca['Cluster'] = kmeans.labels_
    
    plt.figure(figsize=(8,6))
    sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='Cluster', palette='viridis', s=80, alpha=0.8)
    plt.title("Visualización de Clusters de Días Solares (PCA)")
    plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)")
    plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)")
    plt.tight_layout()
    plt.show()


### Interpretación de los Clusters (K-Means)
Calculamos la media de las variables por cada clúster para etiquetarlos conceptualmente.

In [ ]:
if not df_diario.empty:
    perfiles = df_diario.groupby('CLUSTER')[cols_clust].mean().round(2)
    display(perfiles)


## Visualización con t-SNE

El **t-SNE** permite observar vecindarios no lineales. Puntos cercanos representan días operativos muy similares.

In [ ]:
if not df_diario.empty:
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
    X_tsne = tsne.fit_transform(X_scaled)
    
    df_tsne = pd.DataFrame(X_tsne, columns=['Dim1', 'Dim2'])
    df_tsne['Cluster'] = kmeans.labels_
    
    plt.figure(figsize=(8,6))
    sns.scatterplot(data=df_tsne, x='Dim1', y='Dim2', hue='Cluster', palette='viridis', s=70)
    plt.title("Agrupamiento de Días Solares con t-SNE")
    plt.tight_layout()
    plt.show()


# Conclusión

En esta etapa final de **Minería de Datos**, fuimos más allá del simple análisis descriptivo y usamos técnicas avanzadas sobre nuestros datos de telemetría Epever:

1. **Creación de Target (`ALERTA_BATERIA`):** Tradujimos datos crudos a una métrica de negocio útil (salud de la batería).
2. **Combinaciones de Riesgo:** Identificamos discretamente qué condiciones operativas incrementan los problemas de recarga.
3. **Clustering y Reducción de Dimensionalidad (K-Means, PCA, t-SNE):** Segmentamos los cientos de registros en perfiles operativos claros (ej. días de estrés térmico y baja generación vs días óptimos), dándonos herramientas visuales poderosas para modelados predictivos futuros.